# Intro to Maximum Likelihood Estimation

You walk into the living room and find popcorn on the floor. Movie night? Board games? A nap? You instinctively pick the explanation that *makes the evidence most probable* — and that instinct is **maximum likelihood estimation (MLE)**, one of the most widely used ideas for training machine-learning models. Given evidence, compare $P(\text{evidence} \mid \text{scenario})$ across all candidate scenarios and pick the winner. This notebook introduces the idea, quantifies it on coins, and previews how the very same principle chooses regression lines. It is based on the first video of Lesson 6 (Point Estimation — Maximum Likelihood).

## 1. Introduction

MLE is everywhere in machine learning, but the concept behind it is very simple:

> You have some **evidence**, and you want to find the **scenario** that may have led to it. Out of all possible scenarios, pick the one that created the evidence **with the highest probability**.

That's it. The machinery of later videos (likelihood functions, logarithms, calculus) is just this idea made computable.

After completing this notebook you will be able to:

- State the MLE principle: maximize the conditional probability $P(\text{evidence} \mid \text{scenario})$.
- Work the popcorn example and translate it into ML language: maximize $P(\text{data} \mid \text{model})$.
- Compute a first quantitative MLE: which coin most likely produced an observed flip sequence.
- Preview how linear-regression fitting is an instance of the same principle.

## 2. Theory

### The popcorn puzzle

Popcorn lies on the floor next to the couch. Three candidate events:

1. people watching a **movie**,
2. people playing **board games**,
3. somebody taking a **nap**.

Which is most likely to have happened? Compare how strongly each scenario *produces* the evidence:

$$
P(\text{popcorn} \mid \text{movie}) = \text{high}
\qquad
P(\text{popcorn} \mid \text{board games}) = \text{medium}
\qquad
P(\text{popcorn} \mid \text{nap}) = \text{low}
$$

(naps do not produce popcorn on the floor). We go for whatever created the popcorn with the highest probability: **movies**. We *infer* that the most likely thing that happened is a movie night.

### What we actually did

We **maximized a conditional probability**: among $P(\text{popcorn} \mid \text{scenario})$ for all scenarios, we picked the largest. In other words, we found the scenario that most likely *leads to* the evidence. That is **maximum likelihood**: pick the scenario that makes the evidence most likely.

Note the direction: we compared probabilities *of the evidence given each scenario* — not probabilities of the scenarios themselves. (Weighting in how plausible each scenario was beforehand is the Bayesian refinement you met in Lesson 1; MLE deliberately uses the likelihoods alone.)

### The machine-learning translation

This is what's done in machine learning many times over:

- You have a bunch of **data**.
- Several **models** could have generated that data.
- Estimate $P(\text{data} \mid \text{model}_1)$, $P(\text{data} \mid \text{model}_2)$, $P(\text{data} \mid \text{model}_3), \dots$
- **Pick the model with the highest probability** — the model that most likely produced the data:

$$
\text{best model} = \arg\max_{\text{model}} \; P(\text{data} \mid \text{model})
$$

### Preview: where linear regression fits in

Imagine data points scattered on a plane and three candidate lines. Suppose points are *generated near a line* (close to the line = high probability, far = low). Then each line assigns the observed points a probability: $P(\text{points} \mid \text{line}_1)$, etc. The line that gives the points the **highest probability** is the one we pick — and (details in a later video) that choice is exactly least-squares regression.

## 3. Worked Examples

### Example 1: popcorn with numbers

Put rough numbers on the story: $P(\text{popcorn}\mid\text{movie}) = 0.7$, $P(\text{popcorn}\mid\text{games}) = 0.3$, $P(\text{popcorn}\mid\text{nap}) = 0.05$. The argmax is the movie — and notice the numbers don't sum to 1 and don't need to: they are three *different* conditional distributions evaluated at the same evidence, not one distribution over scenarios.

### Example 2: which coin produced HHT?

You observe the flip sequence **H, H, T** and know it came from one of three coins:

| coin | $p$ (heads) | $P(\text{HHT} \mid \text{coin}) = p^2(1-p)$ |
|---|---|---|
| A | 0.3 | $0.09 \cdot 0.7 = 0.063$ |
| B | 0.5 | $0.25 \cdot 0.5 = 0.125$ |
| C | 0.7 | $0.49 \cdot 0.3 = 0.147$ |

Coin **C** made the evidence most probable → MLE picks C. (And if *any* $p$ were allowed, the best would be $p = \tfrac23$ — the sample proportion of heads! A coming attraction.)

### Example 3: evidence changes the verdict

Same three coins, new evidence **T, T, T**: likelihoods $(1-p)^3$ are $0.343, 0.125, 0.027$ — now coin **A** wins. MLE has no loyalty; it re-ranks scenarios every time the evidence changes.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The popcorn decision, as code

In [ ]:
scenarios = {"movie": 0.7, "board games": 0.3, "nap": 0.05}

print("P(popcorn | scenario):")
for s, p in scenarios.items():
    print(f"  {s:<12} {p}")

best = max(scenarios, key=scenarios.get)
print(f"\nmaximum likelihood scenario: {best}")

### Which coin produced HHT?

In [ ]:
def likelihood(flips, p):
    heads = flips.count("H")
    tails = flips.count("T")
    return p**heads * (1 - p)**tails

coins = {"A (p=0.3)": 0.3, "B (p=0.5)": 0.5, "C (p=0.7)": 0.7}

for evidence in ["HHT", "TTT"]:
    print(f"evidence = {evidence}:")
    for name, p in coins.items():
        print(f"  P({evidence} | {name}) = {likelihood(evidence, p):.3f}")
    best = max(coins, key=lambda c: likelihood(evidence, coins[c]))
    print(f"  -> MLE picks coin {best}\n")

### MLE checks out in the long run

Simulate: generate flip sequences from a known coin, run the MLE decision on each, and count how often it recovers the truth. (Watch for a subtlety: accuracy generally grows with evidence, but at very small $n$ the discrete decision thresholds can produce lucky wobbles — $n=3$ happens to sit favorably here.)

In [ ]:
rng = np.random.default_rng(0)

true_p = 0.7
p_candidates = np.array([0.3, 0.5, 0.7])

for n_flips in [3, 10, 50]:
    flips = rng.random(size=(20_000, n_flips)) < true_p
    heads = flips.sum(axis=1)
    # log-likelihood of each candidate for every sequence (log avoids tiny numbers)
    ll = (heads[:, None] * np.log(p_candidates)
          + (n_flips - heads)[:, None] * np.log(1 - p_candidates))
    picked = p_candidates[np.argmax(ll, axis=1)]
    print(f"n = {n_flips:>2} flips: MLE picks the true coin (p=0.7) "
          f"{np.mean(picked == true_p):.1%} of the time")

### The regression preview: three lines, one dataset

Points generated near a hidden line; model = "points fall around the line with Gaussian noise". Each candidate line assigns the data a (log-)likelihood — the best-fitting line wins.

In [ ]:
rng = np.random.default_rng(7)

x = np.linspace(0, 10, 25)
y = 1.5 * x + 2 + rng.normal(0, 1.5, size=len(x))     # truth: slope 1.5, intercept 2

lines = {"line 1: y = 0.5x + 6": (0.5, 6),
         "line 2: y = 1.5x + 2": (1.5, 2),
         "line 3: y = 2.5x - 3": (2.5, -3)}

noise_sd = 1.5
def log_likelihood_line(slope, intercept):
    residuals = y - (slope * x + intercept)
    return np.sum(-0.5 * (residuals / noise_sd)**2
                  - np.log(noise_sd * np.sqrt(2 * np.pi)))

for name, (m, b) in lines.items():
    print(f"log P(data | {name}) = {log_likelihood_line(m, b):>9.1f}")
best = max(lines, key=lambda k: log_likelihood_line(*lines[k]))
print(f"\nMLE picks {best} -- the line the points most likely came from")

## 5. Visualizations

### The popcorn bar chart

In [ ]:
plt.figure(figsize=(7.5, 4))
names = list(scenarios)
values = [scenarios[s] for s in names]
colors = ["tab:green" if s == "movie" else "tab:blue" for s in names]
plt.bar(names, values, color=colors, edgecolor="black", alpha=0.85)
plt.title("P(popcorn on the floor | scenario): pick the tallest bar")
plt.ylabel("conditional probability")
plt.grid(True, alpha=0.3, axis="y")
plt.show()

### The likelihood curve for HHT — all coins at once

Evaluate $P(\text{HHT} \mid p) = p^2(1-p)$ for *every* $p$: the three candidate coins are dots on a smooth curve whose peak sits at $p = \tfrac23$. Choosing among all possible coins is next video's game.

In [ ]:
p_grid = np.linspace(0, 1, 400)
lik = p_grid**2 * (1 - p_grid)

plt.figure(figsize=(9, 4.5))
plt.plot(p_grid, lik, color="tab:blue", linewidth=2,
         label="$P(HHT \\mid p) = p^2(1-p)$")
for name, p in coins.items():
    plt.plot(p, likelihood("HHT", p), "o", markersize=10, label=f"coin {name}")
plt.axvline(2/3, color="gray", linestyle=":",
            label="peak at p = 2/3 (the sample proportion!)")
plt.title("The likelihood of the evidence as a function of the scenario")
plt.xlabel("coin's heads probability p")
plt.ylabel("likelihood of HHT")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### Three candidate lines and the data they must explain

In [ ]:
plt.figure(figsize=(9, 5.5))
plt.scatter(x, y, s=45, color="black", zorder=3, label="data")
grid = np.linspace(0, 10, 100)
for (name, (m, b)), color in zip(lines.items(), ["tab:red", "tab:green", "tab:orange"]):
    ll = log_likelihood_line(m, b)
    plt.plot(grid, m * grid + b, color=color, linewidth=2,
             label=f"{name}  (log-lik {ll:.0f})")
plt.title("Which line most likely generated these points? MLE says: line 2")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The whole story on one page

Everything above in a single sheet: the popcorn scenarios, the coin contest for two kinds of evidence, the likelihood curve over every coin, and the link to machine learning.

<img src="img/01_intro_mle.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** You find muddy paw prints in the kitchen. Scenarios: the dog came in ($P(\text{prints}\mid\text{dog}) = 0.8$), the cat came in ($0.4$), nobody came in ($0.01$). What does MLE conclude? What familiar everyday reasoning does this formalize?

**Exercise 2.** For the evidence **HTHT** and the coins A/B/C above, compute the three likelihoods and find the MLE choice. Why does the fair coin win for this sequence?

### Intermediate

**Exercise 3.** A die is either fair or loaded ($P(6) = 0.5$, other faces $0.1$ each). You roll: 6, 6, 3, 6. Compute the likelihood of the rolls under each die and give the MLE verdict. How would the verdict change if the third roll were another 6?

**Exercise 4.** Likelihoods are not probabilities of scenarios: in Exercise 3, compute the two likelihoods and confirm they don't sum to 1. Then — Bayes callback! — suppose loaded dice are rare: $P(\text{loaded}) = 0.01$. Compute the *posterior* $P(\text{loaded} \mid \text{rolls})$ and compare the conclusion with plain MLE. What did the prior change?

### Advanced

**Exercise 5.** Sweep the evidence: for sequences of $n$ flips containing $k$ heads, the MLE among coins {0.3, 0.5, 0.7} depends only on $k/n$. For $n = 10$, find (numerically) which $k$ values make each coin win, and locate the decision boundaries. Where do they sit relative to the midpoints 0.4 and 0.6?

**Exercise 6.** Estimate a line the MLE way: instead of three fixed candidate lines, do a grid search over slopes and intercepts, maximizing the Gaussian log-likelihood of the regression data above. Compare the MLE line with the true (1.5, 2) — and with `np.polyfit(x, y, 1)`, which minimizes squared error. Why do the two agree? (That's the next videos' punchline.)

## 7. Solutions

In [ ]:
# Exercise 1: paw prints
paw = {"dog": 0.8, "cat": 0.4, "nobody": 0.01}
print(f"Exercise 1: MLE concludes: the {max(paw, key=paw.get)} came in.")
print("This formalizes ordinary abduction -- 'inference to the best explanation':")
print("we habitually blame whichever cause makes the observed mess most likely.")

In [ ]:
# Exercise 2: HTHT
for name, p in coins.items():
    print(f"P(HTHT | {name}) = {likelihood('HTHT', p):.4f}")
print("The fair coin wins: with 2 heads in 4 flips the sample proportion is 0.5,")
print("and p^2(1-p)^2 is maximized at p = 0.5 -- balanced evidence favors the")
print("balanced coin.")

In [ ]:
# Exercise 3: fair vs loaded die
rolls = [6, 6, 3, 6]
def die_likelihood(rolls, p6):
    other = (1 - p6) / 5
    out = 1.0
    for r in rolls:
        out *= p6 if r == 6 else other
    return out

L_fair = die_likelihood(rolls, 1/6)
L_loaded = die_likelihood(rolls, 0.5)
print(f"Exercise 3: P(rolls | fair)   = {L_fair:.6f}")
print(f"            P(rolls | loaded) = {L_loaded:.6f}")
print(f"MLE verdict: {'loaded' if L_loaded > L_fair else 'fair'}")
L_loaded4 = die_likelihood([6, 6, 6, 6], 0.5)
L_fair4 = die_likelihood([6, 6, 6, 6], 1/6)
print(f"with four 6s: loaded/fair likelihood ratio = {L_loaded4/L_fair4:.0f}x --")
print("the verdict is the same but vastly more decisive.")

In [ ]:
# Exercise 4: likelihood vs posterior
print(f"Exercise 4: likelihoods sum to {L_fair + L_loaded:.6f} -- not 1, and they")
print("needn't: they are P(evidence|scenario), not P(scenario).")

prior_loaded = 0.01
posterior_loaded = (L_loaded * prior_loaded) / (L_loaded * prior_loaded
                                                + L_fair * (1 - prior_loaded))
print(f"posterior P(loaded | rolls) = {posterior_loaded:.3f}")
print("Plain MLE says 'loaded'; with the 1% prior the posterior is only ~14%:")
print("rare scenarios need stronger evidence. MLE is the special case of Bayes")
print("with all scenarios treated as equally plausible a priori.")

In [ ]:
# Exercise 5: decision boundaries at n = 10
n = 10
p_candidates = np.array([0.3, 0.5, 0.7])
winners = []
for k in range(n + 1):
    ll = k * np.log(p_candidates) + (n - k) * np.log(1 - p_candidates)
    winners.append(p_candidates[np.argmax(ll)])
print("Exercise 5: k heads -> MLE coin")
for k, w in enumerate(winners):
    print(f"  k = {k:>2}: p = {w}")
print("Boundaries fall between k = 3/4 and k = 6/7, i.e. near k/n ~ 0.39 and 0.61")
print("-- close to (but not exactly at) the midpoints, because likelihood ratios")
print("compare log-odds rather than plain distances between p's.")

In [ ]:
# Exercise 6: grid-search MLE for the line
slopes = np.linspace(0, 3, 301)
intercepts = np.linspace(-5, 9, 281)
best_ll, best_mb = -np.inf, None
for m in slopes:
    resid = y[None, :] - (m * x[None, :] + intercepts[:, None])
    ll = np.sum(-0.5 * (resid / noise_sd)**2, axis=1)
    i = np.argmax(ll)
    if ll[i] > best_ll:
        best_ll, best_mb = ll[i], (m, intercepts[i])

fit = np.polyfit(x, y, 1)
print(f"Exercise 6: MLE grid search: slope {best_mb[0]:.2f}, intercept {best_mb[1]:.2f}")
print(f"np.polyfit (least squares): slope {fit[0]:.2f}, intercept {fit[1]:.2f}")
print(f"truth: slope 1.50, intercept 2.00")
print("Identical (up to grid resolution): with Gaussian noise, maximizing the")
print("likelihood = minimizing the sum of squared residuals -- the equivalence")
print("the coming videos will derive.")

## 8. Conclusion

- **Maximum likelihood estimation**: given evidence, pick the scenario that produced it with the highest probability — maximize the conditional probability $P(\text{evidence} \mid \text{scenario})$.
- The popcorn story is the whole idea: movies > board games > nap at producing popcorn, so movies is the inferred explanation.
- In machine-learning terms: among candidate models, pick $\arg\max P(\text{data} \mid \text{model})$ — the model that most likely generated the data (which coin produced HHT → the $p{=}0.7$ coin; among all $p$, the sample proportion $\tfrac23$).
- Likelihoods are *not* probabilities of the scenarios (they needn't sum to 1); folding in prior plausibility turns MLE into the Bayesian posterior from Lesson 1 — MLE is Bayes with a flat prior.
- Preview delivered: with a Gaussian model of points-near-a-line, the maximum-likelihood line *is* the least-squares regression line — a claim the next videos will make precise.

Next: making the coin example rigorous — the likelihood function of Bernoulli data and its maximization.